#### Imports

In [2]:
import numpy as np
import pandas as pd

#### Read and Display Data

In [5]:
path = ('/Users/devin/Documents/eco-726/nyc-speed-cameras/qwnf7-osfstorage-archive/Speed_900ft_11_22_24.csv')
df = pd.read_csv(path)

#### Perform aggregations and transformations

In [7]:
# parse date columns
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
df['first_appearance'] = pd.to_datetime(df['first_appearance'], errors='coerce')

df['year_month'] = df['Date'].dt.to_period('M').dt.to_timestamp()
df['first_appearance_month'] = df['first_appearance'].dt.to_period('M')

# aggregate monthly (pandas sum() automatically ignores NaNs, matching na.rm = TRUE)
df_monthly = (df.groupby(['ID', 'year_month'], as_index=False)
              .agg(
                  total_collisions=('num_collisions', 'sum')
                  , total_injuries=('total_injuries', 'sum')
                  , first_appearance=('first_appearance_month', 'first'),)
              .fillna({'total_collisions': 0, 'total_injuries': 0}))

# cast date column
df_monthly['first_appearance'] = df_monthly['first_appearance'].dt.to_timestamp()

# create integer indices
df_monthly['t'] = df_monthly['year_month'].dt.strftime('%Y%m').astype(int)
df_monthly['G'] = (df_monthly['first_appearance'].dt.strftime('%Y%m').astype(float).astype('Int64'))

# log transform event counts
df_monthly['log_total_collisions'] = np.log1p(df_monthly['total_collisions'])
df_monthly['log_total_injuries'] = np.log1p(df_monthly['total_injuries'])

# filter out groups (< 10 events)
group_counts = (df_monthly.drop_duplicates(subset=['ID', 'G']).groupby('G').size().reset_index(name='n'))
small_groups = group_counts[group_counts["n"] < 10]['G']
df_monthly = df_monthly[~df_monthly["G"].isin(small_groups)].copy()

df_monthly

,ID,year_month,total_collisions,total_injuries,first_appearance,t,G,log_total_collisions,log_total_injuries
0,1,2012-07-01,1,0,2019-12-01,201207,201912,0.693147,0.000000
1,1,2012-08-01,1,0,2019-12-01,201208,201912,0.693147,0.000000
2,1,2012-09-01,0,0,2019-12-01,201209,201912,0.000000,0.000000
3,1,2012-10-01,1,1,2019-12-01,201210,201912,0.693147,0.693147
4,1,2012-11-01,0,0,2019-12-01,201211,201912,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...
252310,2465,2023-05-01,2,2,2021-11-01,202305,202111,1.098612,1.098612
252311,2465,2023-06-01,1,0,2021-11-01,202306,202111,0.693147,0.000000
252312,2465,2023-07-01,1,1,2021-11-01,202307,202111,0.693147,0.693147
252313,2465,2023-08-01,2,1,2021-11-01,202308,202111,1.098612,0.693147


#### Export CSV

In [9]:
path = ('/Users/devin/Documents/eco-726/model_data/speed900monthly.csv')
df_monthly.to_csv(path, index=False)